In [20]:
from scipy.fftpack import fft, dct
import numpy as np
fft(np.array([4., 3., 5., 10., 5., 3.])).real
dct(np.array([4., 3., 5., 10.]), 1)

array([30., -8.,  6., -2.])

In [21]:
def dct2d(x):
  return fftpack.dct(fftpack.dct(x.T, norm='ortho').T, norm='ortho')
def idct2d(x):
 return fftpack.idct(fftpack.idct(x.T, norm='ortho').T, norm='ortho')

In [22]:
def doDCT(ch, th1, th2, display):
    sh = ch.shape
    v = getVarTh(ch)  #get variance in 32x32 blocks
    thresholds = [np.percentile(v,th1), np.percentile(v,th2)]
    dctSize = [[-1 for j in range(sh[1]/32)] for i in range(sh[0]/32)]
    blkSize = {0:8, 1:16, 2:32}
    coefflist = [[None for j in range(sh[1]/32)] for i in range(sh[0]/32)]
    for i in range(sh[0]/32):  #outer loop for rows
        for j in range(sh[1]/32):  #inner loop for cols
            blk = ch[32*i:32*(i+1), 32*j:32*(j+1)].astype('float32')
            if v[i][j] > thresholds[1]: #use 8x8  #depending on variance, choose DCT block size
                dctSize[i][j] = 0
            elif v[i][j] > thresholds[0]:  #use 16x16
                dctSize[i][j] = 1
            else:  #use 32x32
                dctSize[i][j] = 2
            numsubblocks = 32/blkSize[dctSize[i][j]]; subblocksize = blkSize[dctSize[i][j]]
            coeffs = []
            for kk in range(numsubblocks):  #perform DCT on 1 32x32 block or 4 16x16 blocks or 16 8x8 blocks
                for mm in range(numsubblocks):
                     coeffs += [dct2d(blk[kk*subblocksize:(kk+1)*subblocksize,mm*subblocksize:(mm+1)*subblocksize])]
                coefflist[i][j] = coeffs
    return coefflist, dctSize

In [23]:
def getS(Q):
  return 5000/Q if Q<50 else (200-2*Q if Q!=100 else 1)

In [24]:
def expandQMtx(q, factor):  #function to expand 8x8 quantization matrix to 16x16 or 32x32
    newQ = np.zeros((8*(2**factor), 8*(2**factor)))
    for i in range(8):
        for j in range(8):
            newQ[i*(2**factor):(i+1)*2**factor, j*(2**factor):(j+1)*2**factor] = q[i][j]
    return newQ

In [25]:
def quantize(coefflist, baseMtx, Q): #Quantisizer
    s = getS(Q)
    qMtx = baseMtx*(s/1000.)
    qDict = {1: expandQMtx(qMtx, 2), 4: expandQMtx(qMtx, 1), 16: qMtx}  #expand the base matrix for 16x16 and 32x32 case
    roundedCoeff = [[None for j in range(len(coefflist[0]))] for i in range(len(coefflist))]
    for i in range(len(coefflist)): #For each of the 32x32 blocks
        for j in range(len(coefflist[0])):
            cff = coefflist[i][j]
            q = qDict[len(cff)]
            roundedCoeff[i][j] = [np.round(np.divide(k,q)).astype(int) for k in cff]  #quantization step
    return roundedCoeff 

In [26]:
def getZigzagScanOrder(sz):
    scanOrder = [(0,0)]
    r = 0; c = 0;
    dir = 'b'  #'b' = border, 'd' = down, 'u' = up
    while(len(scanOrder)<sz*sz):
        if dir=='b':  #if at border
            if r==0:  #top border
                c += 1; dir = 'd'
            elif r==sz-1: #bottom border
                c += 1; dir = 'u'
            elif c==0: #left border
                r += 1; dir = 'u'
            elif c==sz-1: #right border
                r += 1; dir = 'd'
            else:
                assert False
        elif dir=='u':  #direction is 'up'
            if r==0 or c==sz-1: #if direction is 'up' nd we are at the border, change mode to 'border'
                dir = 'b'; continue
            else:  #if direction is 'up' nd we are not at the border, go 'up'
                r -= 1; c += 1
        else:
            if c==0 or r==sz-1:  #if direction is 'down' nd we are at the border, change mode to 'border'
                dir = 'b'; continue
            else: #if direction is 'down' nd we are not at the border, go 'down'
                r += 1; c -= 1
        scanOrder.append((r,c))
    return scanOrder